

## Objetivo

Construir un único dataset a partir de tres fuentes:

- **Accidentes:** registros de accidentes por fecha y municipio.
- **Tráfico:** tráfico registrado en peajes por período mensual.
- **Taller:** servicios de reparación registrados por fecha.

El proceso sigue la misma estructura del proyecto anterior:

**Carga → diagnóstico → limpieza → preparación temporal → resumen → integración → transformación → escalamiento → reducción con PCA.**

> Importante: antes de integrar se comprobarán los períodos de cada fuente. Si no existe un período común, el notebook lo mostrará en lugar de inventar una relación temporal.


In [3]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("..")

from src.accidentes import Accidentes
from src.trafico import Trafico
from src.taller import Taller

pd.set_option("display.max_columns", None)


In [5]:
%pip install pandas numpy matplotlib


   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.6 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/9.6 MB 4.6 MB/s eta 0:00:02
   ----------- ---------------------------- 2.9/9.6 MB 5.6 MB/s eta 0:00:02
   ----------------- ---------------------- 4.2/9.6 MB 5.8 MB/s eta 0:00:01
   ---------------------- ----------------- 5.5/9.6 MB 6.0 MB/s eta 0:00:01
   ----------------------------- ---------- 7.1/9.6 MB 6.2 MB/s eta 0:00:01
   ---------------------------------- ----- 8.4/9.6 MB 6.3 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.6 MB 6.3 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 6.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Cargar los tres datasets

Cambia únicamente los nombres de los archivos por los nombres reales que tengas dentro de `dataset/`.

In [5]:
accidentes = Accidentes("../dataset/accidentes.csv")
trafico = Trafico("../dataset/trafico.csv")
taller = Taller("../dataset/taller.csv")

df_accidentes = accidentes.cargar()
df_trafico = trafico.cargar()
df_taller = taller.cargar()

print("Accidentes:", df_accidentes.shape)
print("Tráfico:", df_trafico.shape)
print("Taller:", df_taller.shape)


Accidentes: (633298, 6)
Tráfico: (149944, 9)
Taller: (2000000, 28)


## 2. Diagnóstico inicial

In [6]:
print("===== ACCIDENTES =====")
accidentes.diagnosticar()

print("\n===== TRÁFICO =====")
trafico.diagnosticar()

print("\n===== TALLER =====")
taller.diagnosticar()


===== ACCIDENTES =====
Dimensiones: (633298, 6)

Columnas:
['FECHA HECHO', 'COD_DEPTO', 'DEPARTAMENTO', 'COD_MUNI', 'MUNICIPIO', 'CANTIDAD']

Tipos de datos:
FECHA HECHO       str
COD_DEPTO       int64
DEPARTAMENTO      str
COD_MUNI        int64
MUNICIPIO         str
CANTIDAD        int64
dtype: object

Valores nulos:
FECHA HECHO     0
COD_DEPTO       0
DEPARTAMENTO    0
COD_MUNI        0
MUNICIPIO       0
CANTIDAD        0
dtype: int64

Duplicados: 287996

===== TRÁFICO =====
Dimensiones: (149944, 9)

Columnas:
['IdPeaje', 'Peaje', 'CategoriaTarifa', 'Desde', 'Hasta', 'ValorTarifa', 'CantidadTrafico', 'CantidadEvasores', 'CantidadExentos787']

Tipos de datos:
IdPeaje               int64
Peaje                   str
CategoriaTarifa         str
Desde                   str
Hasta                   str
ValorTarifa           int64
CantidadTrafico       int64
CantidadEvasores      int64
CantidadExentos787    int64
dtype: object

Valores nulos:
IdPeaje               0
Peaje                 0
C

## 3. Limpieza de los tres datasets

In [7]:
accidentes.limpiar()
trafico.limpiar()
taller.limpiar()

print("ACCIDENTES LIMPIOS")
display(accidentes.df.head())

print("TRÁFICO LIMPIO")
display(trafico.df.head())

print("TALLER LIMPIO")
display(taller.df.head())


c:\Users\15-FC0008LA\OneDrive\Documentos\Proyecto_Autogest\autogest\analisis\..\src\trafico.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  self.df["DESDE"] = pd.to_datetime(
c:\Users\15-FC0008LA\OneDrive\Documentos\Proyecto_Autogest\autogest\analisis\..\src\trafico.py:38: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  self.df["HASTA"] = pd.to_datetime(


ACCIDENTES LIMPIOS


,FECHA HECHO,COD_DEPTO,DEPARTAMENTO,COD_MUNI,MUNICIPIO,CANTIDAD
0,2026-07-31,5,ANTIOQUIA,5147,CAREPA,1
1,2026-07-31,76,VALLE DEL CAUCA,76147,CARTAGO,1
3,2026-07-31,76,VALLE DEL CAUCA,76248,EL CERRITO,1
5,2026-07-31,18,CAQUETA,18001,FLORENCIA,1
7,2026-07-31,50,META,50287,FUENTE DE ORO,1


TRÁFICO LIMPIO


,IdPeaje,Peaje,CategoriaTarifa,Desde,Hasta,ValorTarifa,CantidadTrafico,CantidadEvasores,CantidadExentos787,DESDE,HASTA,MES
0,1,ALVARADO,I,2015 Aug 01 12:00:00 AM,2015 Aug 31 12:00:00 AM,7000,27730,0,660,2015-08-01,2015-08-31,2015-08
1,1,ALVARADO,II,2015 Aug 01 12:00:00 AM,2015 Aug 31 12:00:00 AM,7600,9930,0,70,2015-08-01,2015-08-31,2015-08
2,1,ALVARADO,III,2015 Aug 01 12:00:00 AM,2015 Aug 31 12:00:00 AM,16100,1277,0,0,2015-08-01,2015-08-31,2015-08
3,1,ALVARADO,IV,2015 Aug 01 12:00:00 AM,2015 Aug 31 12:00:00 AM,20400,2074,0,0,2015-08-01,2015-08-31,2015-08
4,1,ALVARADO,V,2015 Aug 01 12:00:00 AM,2015 Aug 31 12:00:00 AM,22900,4167,0,0,2015-08-01,2015-08-31,2015-08


TALLER LIMPIO


,Service ID,Customer Name,Vehicle Type,Make and Model,Service Type,Service Description,Repair Date,Towing Date,Tow Location,Drop-off Location,Mechanic Name,Tow Truck Driver,Parts Used,Total Cost,Payment Method,Warranty Information,Service Status,Customer Feedback,Follow-up Needed,Urgency Level,Customer Type,Service Duration Hours,Estimated Cost,Mileage at Service,Insurance Claim Used,Technician Rating,Tow Distance Miles,Is Warranty Valid,FECHA
0,100000,Emily Smith,Bus,BMW School Bus,Oil Change,Diagnosed and repaired Oil Drain Plug.,2022-05-14,NaN,NaN,NaN,Mechanic 2,NaN,"Oil Filter, Oil Drain Plug, Synthetic Oil",77.82,Cash,90 days on labor,In Progress,NaN,Yes,Scheduled,Fleet,3.6,70.50,114248,No,1.9,0.0,Yes,2022-05-14
1,100001,David Smith,Truck,Harley-Davidson Ram 1500,Tire Replacement,Diagnosed and repaired Tire.,2020-07-08,NaN,NaN,NaN,Mechanic 13,NaN,"Rim, Tire, TPMS Sensor",135.97,Insurance,6 months on new tire,s,NaN,No,Emergency,Fleet,1.9,125.00,250870,Yes,4.9,0.0,Yes,2020-07-08
2,100002,James Johnson,Van,BMW Odyssey,Battery Replacement,Diagnosed and repaired Battery.,2023-09-16,NaN,NaN,NaN,Mechanic 8,NaN,"Battery Terminal, Battery, Battery Holder",329.70,Credit Card,1 year on battery,s,NaN,Yes,Standard,Business,3.9,347.71,155738,No,1.7,0.0,Yes,2023-09-16
3,100003,Laura Brown,Bus,BMW School Bus,Battery Replacement,Diagnosed and repaired Battery.,2022-08-04,NaN,NaN,NaN,Mechanic 13,NaN,"Battery Holder, Battery, Battery Terminal",146.93,Company Invoice,1 year on battery,s,NaN,No,Emergency,Individual,0.5,140.17,130087,No,2.6,0.0,Yes,2022-08-04
4,100004,Emily Brown,Van,Dodge Transit,Towing,Diagnosed and repaired no parts used.,2022-10-19,2022-03-21,Dealer,Commercial,Mechanic 5,Driver 5,No parts used,92.68,Company Invoice,No warranty offered,s,NaN,No,Scheduled,Business,4.9,86.00,219400,No,1.9,99.8,No,2022-10-19


## 4. Guardar las versiones limpias

In [8]:
accidentes.guardar("../dataset/accidentes_limpio.csv")
trafico.guardar("../dataset/trafico_limpio.csv")
taller.guardar("../dataset/taller_limpio.csv")

print("Datasets limpios guardados.")


Datasets limpios guardados.


## 5. Revisar los períodos de tiempo

In [9]:
print("PERÍODO ACCIDENTES")
print(accidentes.df["FECHA HECHO"].min(), "a", accidentes.df["FECHA HECHO"].max())

print("\nPERÍODO TRÁFICO")
print(trafico.df["DESDE"].min(), "a", trafico.df["HASTA"].max())

print("\nPERÍODO TALLER")
print(taller.df["FECHA"].min(), "a", taller.df["FECHA"].max())


PERÍODO ACCIDENTES
2003-01-01 00:00:00 a 2026-07-31 00:00:00

PERÍODO TRÁFICO
2014-01-01 00:00:00 a 2026-05-31 00:00:00

PERÍODO TALLER
2020-01-01 00:00:00 a 2024-12-31 00:00:00


### ¿Por qué revisar el período?

Los tres datasets no tienen necesariamente la misma unidad temporal:

- Accidentes: día.
- Taller: día.
- Tráfico: período de peaje, en el ejemplo un mes.

Por eso primero se debe verificar que exista coincidencia temporal real.


## 6. Crear resúmenes por fecha

In [10]:
# Accidentes por día
accidentes_dia = (
    accidentes.df
    .groupby("FECHA HECHO")
    .agg(
        ACCIDENTES=("CANTIDAD", "sum"),
        MUNICIPIOS_ACCIDENTES=("MUNICIPIO", "nunique")
    )
    .reset_index()
)

# Taller por día
taller_dia = (
    taller.df
    .groupby("FECHA")
    .agg(
        SERVICIOS_TALLER=("Service ID", "count"),
        COSTO_TALLER=("Total Cost", "sum"),
        DURACION_PROMEDIO_TALLER=("Service Duration Hours", "mean"),
        KILOMETRAJE_PROMEDIO=("Mileage at Service", "mean")
    )
    .reset_index()
)

display(accidentes_dia.head())
display(taller_dia.head())


,FECHA HECHO,ACCIDENTES,MUNICIPIOS_ACCIDENTES
0,2003-01-01,28,28
1,2003-01-02,16,16
2,2003-01-03,16,16
3,2003-01-04,18,16
4,2003-01-05,22,20


,FECHA,SERVICIOS_TALLER,COSTO_TALLER,DURACION_PROMEDIO_TALLER,KILOMETRAJE_PROMEDIO
0,2020-01-01,989,394778.04,3.028008,153955.522750
1,2020-01-02,1074,386474.62,2.973277,155409.973929
2,2020-01-03,1095,430199.94,3.094977,154927.003653
3,2020-01-04,1103,396071.88,2.961378,157889.802357
4,2020-01-05,1122,405669.71,3.035116,152733.056150


## 7. Resumen mensual del tráfico

In [11]:
trafico_mes = (
    trafico.df
    .groupby("MES")
    .agg(
        TRAFICO=("CantidadTrafico", "sum"),
        EVASORES=("CantidadEvasores", "sum"),
        EXENTOS=("CantidadExentos787", "sum"),
        PEAJES=("IdPeaje", "nunique"),
        TARIFA_PROMEDIO=("ValorTarifa", "mean")
    )
    .reset_index()
)

display(trafico_mes.head())


,MES,TRAFICO,EVASORES,EXENTOS,PEAJES,TARIFA_PROMEDIO
0,2014-01,20193725,32,161367,88,14545.894040
1,2014-02,15440249,30,156040,88,14477.212682
2,2014-03,17218754,32,174424,88,14425.164258
3,2014-04,18157033,32,167009,88,14426.741130
4,2014-05,16965456,41,174638,88,14441.963109


## 8. Preparar la llave temporal

Como tráfico está expresado por períodos mensuales, se creará una columna `MES` en accidentes y taller.

Esto permite relacionar:

**día de accidente/taller → mes correspondiente → tráfico del mes.**

No se debe interpretar el tráfico mensual como si fuera el tráfico exacto de cada día.


In [12]:
accidentes_dia["MES"] = accidentes_dia["FECHA HECHO"].dt.to_period("M").astype(str)
taller_dia["MES"] = taller_dia["FECHA"].dt.to_period("M").astype(str)

print("Meses accidentes:", accidentes_dia["MES"].nunique())
print("Meses tráfico:", trafico_mes["MES"].nunique())
print("Meses taller:", taller_dia["MES"].nunique())

meses_comunes = (
    set(accidentes_dia["MES"])
    & set(trafico_mes["MES"])
    & set(taller_dia["MES"])
)

print("Meses presentes en los tres datasets:", len(meses_comunes))
print(sorted(meses_comunes)[:10])


Meses accidentes: 283
Meses tráfico: 149
Meses taller: 60
Meses presentes en los tres datasets: 60
['2020-01', '2020-02', '2020-03', '2020-04', '2020-05', '2020-06', '2020-07', '2020-08', '2020-09', '2020-10']


## 9. Integrar los tres datasets

In [13]:
print("ACCIDENTES_DIA:")
print(accidentes_dia.columns.tolist())

print("\nTALLER_DIA:")
print(taller_dia.columns.tolist())

print("\nTRAFICO_MES:")
print(trafico_mes.columns.tolist())


ACCIDENTES_DIA:
['FECHA HECHO', 'ACCIDENTES', 'MUNICIPIOS_ACCIDENTES', 'MES']

TALLER_DIA:
['FECHA', 'SERVICIOS_TALLER', 'COSTO_TALLER', 'DURACION_PROMEDIO_TALLER', 'KILOMETRAJE_PROMEDIO', 'MES']

TRAFICO_MES:
['MES', 'TRAFICO', 'EVASORES', 'EXENTOS', 'PEAJES', 'TARIFA_PROMEDIO']


In [14]:
dataset_integrado = (
    accidentes_dia
    .merge(
        taller_dia,
        left_on=["FECHA HECHO", "MES"],
        right_on=["FECHA", "MES"],
        how="outer"
    )
    .merge(
        trafico_mes,
        on="MES",
        how="left"
    )
    .sort_values("FECHA HECHO")
    .reset_index(drop=True)
)

print("Tamaño inicial:", dataset_integrado.shape)
display(dataset_integrado.head(10))


Tamaño inicial: (8613, 14)


,FECHA HECHO,ACCIDENTES,MUNICIPIOS_ACCIDENTES,MES,FECHA,SERVICIOS_TALLER,COSTO_TALLER,DURACION_PROMEDIO_TALLER,KILOMETRAJE_PROMEDIO,TRAFICO,EVASORES,EXENTOS,PEAJES,TARIFA_PROMEDIO
0,2003-01-01,28,28,2003-01,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2003-01-02,16,16,2003-01,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2003-01-03,16,16,2003-01,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2003-01-04,18,16,2003-01,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2003-01-05,22,20,2003-01,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2003-01-06,25,23,2003-01,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2003-01-07,19,19,2003-01,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2003-01-08,17,12,2003-01,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2003-01-09,15,15,2003-01,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2003-01-10,24,16,2003-01,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
print("ACCIDENTES:")
print(accidentes_dia["FECHA HECHO"].min())
print(accidentes_dia["FECHA HECHO"].max())

print("\nTALLER:")
print(taller_dia["FECHA"].min())
print(taller_dia["FECHA"].max())


ACCIDENTES:
2003-01-01 00:00:00
2026-07-31 00:00:00

TALLER:
2020-01-01 00:00:00
2024-12-31 00:00:00


In [16]:
print("\nPrimeras fechas de accidentes:")
print(accidentes_dia["FECHA HECHO"].head())

print("\nPrimeras fechas de taller:")
print(taller_dia["FECHA"].head())



Primeras fechas de accidentes:
0   2003-01-01
1   2003-01-02
2   2003-01-03
3   2003-01-04
4   2003-01-05
Name: FECHA HECHO, dtype: datetime64[us]

Primeras fechas de taller:
0   2020-01-01
1   2020-01-02
2   2020-01-03
3   2020-01-04
4   2020-01-05
Name: FECHA, dtype: datetime64[us]


In [17]:
print(accidentes_dia["FECHA HECHO"].dtype)
print(taller_dia["FECHA"].dtype)


datetime64[us]
datetime64[us]


In [18]:
dataset_integrado = (
    accidentes_dia
    .merge(
        taller_dia,
        left_on=["FECHA HECHO", "MES"],
        right_on=["FECHA", "MES"],
        how="outer"
    )
    .merge(
        trafico_mes,
        on="MES",
        how="left"
    )
)

# Crear una fecha única
dataset_integrado["FECHA_FINAL"] = (
    dataset_integrado["FECHA HECHO"]
    .combine_first(dataset_integrado["FECHA"])
)

# Eliminar las fechas originales
dataset_integrado = dataset_integrado.drop(
    columns=["FECHA HECHO", "FECHA"]
)

# Renombrar la fecha final
dataset_integrado = dataset_integrado.rename(
    columns={"FECHA_FINAL": "FECHA"}
)

# Ordenar
dataset_integrado = (
    dataset_integrado
    .sort_values("FECHA")
    .reset_index(drop=True)
)

print("Tamaño del dataset integrado:", dataset_integrado.shape)

display(dataset_integrado.head(10))



Tamaño del dataset integrado: (8613, 13)


,ACCIDENTES,MUNICIPIOS_ACCIDENTES,MES,SERVICIOS_TALLER,COSTO_TALLER,DURACION_PROMEDIO_TALLER,KILOMETRAJE_PROMEDIO,TRAFICO,EVASORES,EXENTOS,PEAJES,TARIFA_PROMEDIO,FECHA
0,28,28,2003-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2003-01-01
1,16,16,2003-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2003-01-02
2,16,16,2003-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2003-01-03
3,18,16,2003-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2003-01-04
4,22,20,2003-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2003-01-05
5,25,23,2003-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2003-01-06
6,19,19,2003-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2003-01-07
7,17,12,2003-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2003-01-08
8,15,15,2003-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2003-01-09
9,24,16,2003-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2003-01-10


## 10. Revisar datos faltantes después de la integración

Los valores faltantes son importantes porque pueden significar que un dataset no tenía registro para esa fecha, no necesariamente que el valor fuera cero.


In [19]:
print(dataset_integrado.isnull().sum())


ACCIDENTES                     0
MUNICIPIOS_ACCIDENTES          0
MES                            0
SERVICIOS_TALLER            6786
COSTO_TALLER                6786
DURACION_PROMEDIO_TALLER    6786
KILOMETRAJE_PROMEDIO        6786
TRAFICO                     4079
EVASORES                    4079
EXENTOS                     4079
PEAJES                      4079
TARIFA_PROMEDIO             4079
FECHA                          0
dtype: int64


## 11. Conservar fechas con información de accidentes, taller o tráfico

In [20]:
dataset_integrado = dataset_integrado[
    dataset_integrado["ACCIDENTES"].notna()
    | dataset_integrado["SERVICIOS_TALLER"].notna()
    | dataset_integrado["TRAFICO"].notna()
].copy()

dataset_integrado = dataset_integrado.sort_values("FECHA").reset_index(drop=True)

print("Tamaño final de integración:", dataset_integrado.shape)
display(dataset_integrado.head())


Tamaño final de integración: (8613, 13)


,ACCIDENTES,MUNICIPIOS_ACCIDENTES,MES,SERVICIOS_TALLER,COSTO_TALLER,DURACION_PROMEDIO_TALLER,KILOMETRAJE_PROMEDIO,TRAFICO,EVASORES,EXENTOS,PEAJES,TARIFA_PROMEDIO,FECHA
0,28,28,2003-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2003-01-01
1,16,16,2003-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2003-01-02
2,16,16,2003-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2003-01-03
3,18,16,2003-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2003-01-04
4,22,20,2003-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2003-01-05


12. Discretización de variables numéricas



La discretización transforma variables numéricas continuas en categorías.

En AutoGest se aplicará principalmente a:

COSTO_TALLER
DURACION_PROMEDIO_TALLER

Esto permite convertir valores numéricos en categorías más fáciles de interpretar para el análisis.

Por ejemplo:

Costo:

Bajo
Medio
Alto

Duración:

Corta
Media
Larga

Se utilizará discretización por cuantiles, dividiendo los valores disponibles en tres grupos.

Las columnas originales no se eliminarán. Se conservarán COSTO_TALLER y DURACION_PROMEDIO_TALLER, y se crearán nuevas columnas categóricas.

In [21]:
dataset_integrado["CATEGORIA_COSTO"] = pd.qcut(
    dataset_integrado["COSTO_TALLER"],
    q=3,
    labels=["Bajo", "Medio", "Alto"],
    duplicates="drop"
)

dataset_integrado["CATEGORIA_DURACION"] = pd.qcut(
    dataset_integrado["DURACION_PROMEDIO_TALLER"],
    q=3,
    labels=["Corta", "Media", "Larga"],
    duplicates="drop"
)

print("Discretización realizada correctamente.")

Discretización realizada correctamente.


13. Validar la discretización

Se revisará cuántos registros quedaron en cada categoría.

In [22]:
print("===== CATEGORÍA DE COSTO =====")
print(
    dataset_integrado["CATEGORIA_COSTO"]
    .value_counts(dropna=False)
)

print("\n===== CATEGORÍA DE DURACIÓN =====")
print(
    dataset_integrado["CATEGORIA_DURACION"]
    .value_counts(dropna=False)
)

===== CATEGORÍA DE COSTO =====
CATEGORIA_COSTO
NaN      6786
Bajo      609
Medio     609
Alto      609
Name: count, dtype: int64

===== CATEGORÍA DE DURACIÓN =====
CATEGORIA_DURACION
NaN      6786
Corta     609
Media     609
Larga     609
Name: count, dtype: int64


Para visualizar únicamente los registros que sí tienen taller

In [23]:
display(
    dataset_integrado[
        dataset_integrado["COSTO_TALLER"].notna()
    ][
        [
            "FECHA",
            "COSTO_TALLER",
            "CATEGORIA_COSTO",
            "DURACION_PROMEDIO_TALLER",
            "CATEGORIA_DURACION"
        ]
    ].head(20)
)

,FECHA,COSTO_TALLER,CATEGORIA_COSTO,DURACION_PROMEDIO_TALLER,CATEGORIA_DURACION
6209,2020-01-01,394778.04,Bajo,3.028008,Media
6210,2020-01-02,386474.62,Bajo,2.973277,Corta
6211,2020-01-03,430199.94,Alto,3.094977,Larga
6212,2020-01-04,396071.88,Bajo,2.961378,Corta
6213,2020-01-05,405669.71,Medio,3.035116,Media
6214,2020-01-06,441240.77,Alto,3.044309,Media
6215,2020-01-07,398151.58,Bajo,3.033638,Media
6216,2020-01-08,394339.64,Bajo,3.018449,Media
6217,2020-01-09,403579.01,Medio,2.988940,Corta
6218,2020-01-10,369875.58,Bajo,3.041422,Media


14. Comprobar los límites de cada categoría

ste paso es importante para demostrar que la discretización realmente se hizo:

In [24]:
print("===== RANGO DE CADA CATEGORÍA DE COSTO =====")

print(
    dataset_integrado
    .groupby("CATEGORIA_COSTO", observed=True)["COSTO_TALLER"]
    .agg(["min", "max", "count"])
)

print("\n===== RANGO DE CADA CATEGORÍA DE DURACIÓN =====")

print(
    dataset_integrado
    .groupby("CATEGORIA_DURACION", observed=True)["DURACION_PROMEDIO_TALLER"]
    .agg(["min", "max", "count"])
)

===== RANGO DE CADA CATEGORÍA DE COSTO =====
                       min        max  count
CATEGORIA_COSTO                             
Bajo             347462.49  402333.73    609
Medio            402397.20  417163.94    609
Alto             417254.70  476025.17    609

===== RANGO DE CADA CATEGORÍA DE DURACIÓN =====
                         min       max  count
CATEGORIA_DURACION                           
Corta               2.871265  3.008438    609
Media               3.008587  3.045620    609
Larga               3.045622  3.185848    609


In [35]:
print("CATEGORIA_COSTO")
print(dataset_integrado["CATEGORIA_COSTO"].value_counts(dropna=False))

print("\nCATEGORIA_DURACION")
print(dataset_integrado["CATEGORIA_DURACION"].value_counts(dropna=False))

CATEGORIA_COSTO
CATEGORIA_COSTO
NaN      6786
Bajo      609
Alto      609
Medio     609
Name: count, dtype: int64

CATEGORIA_DURACION
CATEGORIA_DURACION
NaN      6786
Media     609
Corta     609
Larga     609
Name: count, dtype: int64


In [36]:
print("\nValores de COSTO_TALLER:")
print(dataset_integrado["COSTO_TALLER"].describe())

print("\nValores de DURACION_PROMEDIO_TALLER:")
print(dataset_integrado["DURACION_PROMEDIO_TALLER"].describe())


Valores de COSTO_TALLER:
count      1827.000000
mean     409655.624992
std       18049.142288
min      347462.490000
25%      397596.305000
50%      409804.270000
75%      421385.420000
max      476025.170000
Name: COSTO_TALLER, dtype: float64

Valores de DURACION_PROMEDIO_TALLER:
count    1827.000000
mean        3.029008
std         0.044194
min         2.871265
25%         2.997747
50%         3.027091
75%         3.057711
max         3.185848
Name: DURACION_PROMEDIO_TALLER, dtype: float64


## 12. Guardar el dataset integrado

In [24]:
dataset_integrado.to_csv(
    "../dataset/dataset_integrado_accidentes_trafico_taller.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Dataset integrado guardado correctamente.")


Dataset integrado guardado correctamente.
